# 10 Multi-Accident Classification

This notebook uses the 09 audit decision: 12 sufficiently sampled accident classes, complete trajectory/sample_id groups, 30/60 s primary windows, and 120 s exploratory coverage. Group A is the 38-variable strict process baseline; B removes the candidate leakage feature identified by 09; C removes the 14 SLBIC initial-condition-confounded candidates.

In [1]:
from pathlib import Path
import json
import sys

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').exists())
sys.path.insert(0, str(PROJECT_ROOT))

from src.multi_accident_features import (
    FIRST_VERSION_CLASSES,
    PRIMARY_WINDOWS_S,
    WINDOWS_S,
    build_feature_dataset,
    feature_columns,
    input_feature_groups,
)
from src.multi_accident_models import (
    evaluate_models,
    selected_test_row,
    select_best_validation,
)

RESULT_ROOT = PROJECT_ROOT / 'results'
FIGURE_ROOT = RESULT_ROOT / 'figures'
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
CLASS_LABELS = list(FIRST_VERSION_CLASSES)


## Audit gates and input groups

In [2]:
feature_groups, removed_features = input_feature_groups()
assert len(feature_groups['A_strict_38']) == 38
assert removed_features['B_without_potential_leakage'] == ['LVCR']
assert len(removed_features['C_without_SLBIC_initial']) == 14

leakage = pd.read_csv(RESULT_ROOT / '09_multi_accident_leakage_flags.csv')
strict_leakage = sorted(set(leakage.loc[leakage['potential_label_leakage'].astype(str).str.lower().eq('true'), 'feature']) & set(feature_groups['A_strict_38']))
assert strict_leakage == ['LVCR']
non_strict_leakage = set(leakage.loc[leakage['potential_label_leakage'].astype(str).str.lower().eq('true'), 'feature']) - set(feature_groups['A_strict_38'])
assert {'WFLB', 'WLR', 'WTRA', 'WTRB', 'WRLA', 'WRLB', 'RM4', 'SGLK', 'STSG'} <= non_strict_leakage

print('Classes:', ', '.join(CLASS_LABELS))
print('Input groups:', {name: len(columns) for name, columns in feature_groups.items()})
print('B removed from strict set:', removed_features['B_without_potential_leakage'])
print('C removed from strict set:', removed_features['C_without_SLBIC_initial'])
print('09 potential leakage rows:', int(leakage['potential_label_leakage'].astype(str).str.lower().eq('true').sum()))


Classes: FLB, LLB, LOCA, LOCAC, LR, MD, RI, RW, SGATR, SGBTR, SLBIC, SLBOC
Input groups: {'A_strict_38': 38, 'B_without_potential_leakage': 37, 'C_without_SLBIC_initial': 24}
B removed from strict set: ['LVCR']
C removed from strict set: ['PSGA', 'PSGB', 'QMGA', 'QMGB', 'QMWT', 'TAVG', 'TCA', 'TCB', 'THA', 'THB', 'WFWA', 'WFWB', 'WSTA', 'WSTB']
09 potential leakage rows: 16


## Window features and grouped dataset

In [3]:
dataset, window_points = build_feature_dataset(
    project_root=PROJECT_ROOT, windows_s=WINDOWS_S, classes=tuple(CLASS_LABELS),
)
inventory = pd.read_csv(RESULT_ROOT / '09_multi_accident_class_inventory.csv')
expected_inventory = inventory.loc[inventory['accident_class'].isin(CLASS_LABELS)]
expected_by_window = {window: int(expected_inventory[f'available_{window}s'].sum()) for window in WINDOWS_S}
actual_by_window = dataset.groupby('window_s')['sample_id'].nunique().to_dict()
assert actual_by_window == expected_by_window
assert actual_by_window[30] == actual_by_window[60]
assert set(dataset['split']) == {'train', 'validation', 'test'}
assert dataset.groupby('sample_id')['split'].nunique().max() == 1
assert not dataset[feature_columns(feature_groups['A_strict_38'])].isna().any().any()

window_points.to_csv(RESULT_ROOT / '10_multi_accident_window_points.csv', index=False)
class_counts = (
    dataset.groupby(['window_s', 'split', 'accident_class'], as_index=False)['sample_id']
    .nunique()
    .rename(columns={'sample_id': 'trajectory_count'})
)
class_counts.to_csv(RESULT_ROOT / '10_multi_accident_class_counts.csv', index=False)
print('Rows by window:', actual_by_window)
display(class_counts.query('window_s in [30, 60]').head(36))


Rows by window: {30: 1211, 60: 1211, 120: 1193}


,window_s,split,accident_class,trajectory_count
0,30,test,FLB,10
1,30,test,LLB,11
2,30,test,LOCA,10
3,30,test,LOCAC,10
4,30,test,LR,10
5,30,test,MD,10
6,30,test,RI,10
7,30,test,RW,10
8,30,test,SGATR,10
9,30,test,SGBTR,11


## Baselines and sensitivity experiments

In [4]:
metrics, per_class, confusion = evaluate_models(
    dataset, feature_groups, CLASS_LABELS, windows_s=WINDOWS_S
)
assert len(metrics) == len(WINDOWS_S) * len(feature_groups) * 4 * 2
assert metrics[['accuracy', 'macro_f1', 'balanced_accuracy']].notna().all().all()
assert set(metrics['eval_split']) == {'validation', 'test'}
assert set(metrics['model']) == {'naive_majority', 'logistic_regression', 'random_forest', 'hist_gradient_boosting'}

metrics.to_csv(RESULT_ROOT / '10_multi_accident_metrics.csv', index=False)
per_class.to_csv(RESULT_ROOT / '10_multi_accident_per_class.csv', index=False)
confusion.to_csv(RESULT_ROOT / '10_multi_accident_confusion_matrices.csv', index=False)

primary_test = metrics.loc[
    metrics['eval_split'].eq('test') & metrics['window_s'].isin(PRIMARY_WINDOWS_S)
].copy()
comparison_30_60 = primary_test.sort_values(['input_group', 'model', 'window_s'])
comparison_30_60.to_csv(RESULT_ROOT / '10_multi_accident_30_vs_60.csv', index=False)

base_lookup = primary_test.loc[primary_test['input_group'].eq('A_strict_38')].set_index(['window_s', 'model'])
sensitivity_rows = []
for row in primary_test.itertuples(index=False):
    base = base_lookup.loc[(row.window_s, row.model)]
    slbic = per_class.loc[
        per_class['window_s'].eq(row.window_s) & per_class['model'].eq(row.model)
        & per_class['input_group'].eq(row.input_group) & per_class['eval_split'].eq('test')
        & per_class['accident_class'].eq('SLBIC'), 'recall'
    ].iloc[0]
    base_slbic = per_class.loc[
        per_class['window_s'].eq(row.window_s) & per_class['model'].eq(row.model)
        & per_class['input_group'].eq('A_strict_38') & per_class['eval_split'].eq('test')
        & per_class['accident_class'].eq('SLBIC'), 'recall'
    ].iloc[0]
    sensitivity_rows.append({
        'window_s': row.window_s, 'model': row.model, 'input_group': row.input_group,
        'test_macro_f1': row.macro_f1, 'test_balanced_accuracy': row.balanced_accuracy,
        'slbic_recall': slbic, 'delta_macro_f1_vs_A': row.macro_f1 - base.macro_f1,
        'delta_balanced_accuracy_vs_A': row.balanced_accuracy - base.balanced_accuracy,
        'delta_slbic_recall_vs_A': slbic - base_slbic,
    })
sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(RESULT_ROOT / '10_multi_accident_sensitivity_comparison.csv', index=False)
display(metrics.query("window_s in [30, 60] and eval_split == 'test'").sort_values('macro_f1', ascending=False).head(18))


,window_s,input_group,model,eval_split,n_samples,n_features,accuracy,macro_f1,balanced_accuracy,train_majority_class
3,30,A_strict_38,logistic_regression,test,123,190,0.276423,0.192755,0.272727,
5,30,A_strict_38,random_forest,test,123,190,0.276423,0.192755,0.272727,
7,30,A_strict_38,hist_gradient_boosting,test,123,190,0.276423,0.192755,0.272727,
11,30,B_without_potential_leakage,logistic_regression,test,123,185,0.276423,0.192755,0.272727,
15,30,B_without_potential_leakage,hist_gradient_boosting,test,123,185,0.276423,0.192755,0.272727,
13,30,B_without_potential_leakage,random_forest,test,123,185,0.276423,0.192755,0.272727,
21,30,C_without_SLBIC_initial,random_forest,test,123,120,0.276423,0.192755,0.272727,
19,30,C_without_SLBIC_initial,logistic_regression,test,123,120,0.276423,0.192755,0.272727,
47,60,C_without_SLBIC_initial,hist_gradient_boosting,test,123,120,0.276423,0.192755,0.272727,
45,60,C_without_SLBIC_initial,random_forest,test,123,120,0.276423,0.192755,0.272727,


## Confusion matrices and comparison figures

In [5]:
def best_test_matrix(input_group, window_s):
    selection = select_best_validation(metrics, input_group=input_group, windows_s=(window_s,))
    test_row = selected_test_row(metrics, selection)
    table = confusion.loc[
        confusion['window_s'].eq(window_s) & confusion['input_group'].eq(input_group)
        & confusion['model'].eq(selection['model']) & confusion['eval_split'].eq('test')
    ]
    matrix = table.pivot(index='true_class', columns='predicted_class', values='count').reindex(
        index=CLASS_LABELS, columns=CLASS_LABELS, fill_value=0
    ).to_numpy()
    return selection, test_row, matrix

for input_group in feature_groups:
    for window_s in WINDOWS_S:
        selection, test_row, matrix = best_test_matrix(input_group, window_s)
        fig, ax = plt.subplots(figsize=(9, 7))
        image = ax.imshow(matrix, cmap='Blues', vmin=0)
        for i in range(len(CLASS_LABELS)):
            for j in range(len(CLASS_LABELS)):
                ax.text(j, i, int(matrix[i, j]), ha='center', va='center', fontsize=7)
        ax.set_xticks(range(len(CLASS_LABELS)), CLASS_LABELS, rotation=60, ha='right')
        ax.set_yticks(range(len(CLASS_LABELS)), CLASS_LABELS)
        ax.set_xlabel('Predicted class')
        ax.set_ylabel('True class')
        ax.set_title(f"Test confusion: {input_group}, {window_s}s, {selection['model']}")
        fig.colorbar(image, ax=ax, label='Count')
        fig.tight_layout()
        fig.savefig(FIGURE_ROOT / f"10_confusion_{input_group}_{window_s}s.png", dpi=150)
        plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for metric_name, ax in [('macro_f1', axes[0]), ('balanced_accuracy', axes[1])] :
    for (group, model), frame in primary_test.groupby(['input_group', 'model']):
        frame = frame.sort_values('window_s')
        ax.plot(frame['window_s'], frame[metric_name], marker='o', label=f'{group} | {model}')
    ax.set_xticks(PRIMARY_WINDOWS_S)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel('Window (s)')
    ax.set_ylabel(metric_name.replace('_', ' ').title())
    ax.grid(alpha=0.25)
axes[0].legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc='upper left')
fig.suptitle('30 s versus 60 s test comparison')
fig.tight_layout()
fig.savefig(FIGURE_ROOT / '10_30_vs_60_comparison.png', dpi=150, bbox_inches='tight')
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
group_order = ['A_strict_38', 'B_without_potential_leakage', 'C_without_SLBIC_initial']
x = np.arange(len(group_order))
for (window_s, model), frame in sensitivity.groupby(['window_s', 'model']):
    frame = frame.set_index('input_group').reindex(group_order)
    label = f'{window_s}s | {model}'
    axes[0].plot(x, frame['delta_macro_f1_vs_A'], marker='o', label=label)
    axes[1].plot(x, frame['delta_slbic_recall_vs_A'], marker='o', label=label)
for ax, ylabel in zip(axes, ['Macro-F1 delta vs A', 'SLBIC recall delta vs A']):
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(x, group_order, rotation=25, ha='right')
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25)
axes[0].legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc='upper left')
fig.suptitle('Leakage and initial-condition sensitivity on test set')
fig.tight_layout()
fig.savefig(FIGURE_ROOT / '10_sensitivity_comparison.png', dpi=150, bbox_inches='tight')
plt.close(fig)


## Summary and assertions

In [6]:
best_A_validation = select_best_validation(metrics, input_group='A_strict_38')
best_A_test = selected_test_row(metrics, best_A_validation)
best_A_120_validation = select_best_validation(metrics, input_group='A_strict_38', windows_s=(120,))
best_A_120_test = selected_test_row(metrics, best_A_120_validation)
best_overall_validation = select_best_validation(metrics)
best_overall_test = selected_test_row(metrics, best_overall_validation)
a_primary = metrics.loc[metrics['input_group'].eq('A_strict_38') & metrics['eval_split'].eq('test') & metrics['window_s'].isin(PRIMARY_WINDOWS_S)]
a_30_60_delta = float(a_primary.pivot(index='model', columns='window_s', values='macro_f1').diff(axis=1).iloc[:, -1].abs().max())
selected_sensitivity = sensitivity.loc[(sensitivity['window_s'].eq(best_A_test['window_s'])) & (sensitivity['model'].eq(best_A_test['model']))].to_dict(orient='records')

best_A_recall = per_class.loc[
    per_class['window_s'].eq(best_A_test['window_s']) & per_class['input_group'].eq(best_A_test['input_group'])
    & per_class['model'].eq(best_A_test['model']) & per_class['eval_split'].eq('test')
].sort_values('recall')
weakest = best_A_recall.head(3)[['accident_class', 'recall', 'support']].to_dict(orient='records')
strongest = best_A_recall.tail(3).sort_values('recall', ascending=False)[['accident_class', 'recall', 'support']].to_dict(orient='records')

best_confusion = confusion.loc[
    confusion['window_s'].eq(best_A_test['window_s']) & confusion['input_group'].eq(best_A_test['input_group'])
    & confusion['model'].eq(best_A_test['model']) & confusion['eval_split'].eq('test')
    & confusion['true_class'].ne(confusion['predicted_class'])
].sort_values('count', ascending=False)
confusion_pairs = best_confusion.head(8)[['true_class', 'predicted_class', 'count']].to_dict(orient='records')

def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, list):
        return [json_safe(item) for item in value]
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if pd.isna(value) if not isinstance(value, (tuple, set)) else False:
        return None
    return value

near_perfect = bool(primary_test['macro_f1'].max() >= 0.95)
summary = {
    'experiment': 'multi-accident classification baseline',
    'classes': CLASS_LABELS,
    'trajectory_counts_by_window': {str(key): int(value) for key, value in actual_by_window.items()},
    'primary_windows_s': list(PRIMARY_WINDOWS_S),
    'exploratory_windows_s': [120],
    'input_groups': {key: {'feature_count': len(value), 'features': value, 'removed_features': removed_features.get(key, [])} for key, value in feature_groups.items()},
    'best_strict_group_A': {'validation_selection': best_A_validation.to_dict(), 'test_metrics': best_A_test.to_dict()},
    'best_overall_primary': {'validation_selection': best_overall_validation.to_dict(), 'test_metrics': best_overall_test.to_dict()},
    'best_exploratory_120_group_A': {'validation_selection': best_A_120_validation.to_dict(), 'test_metrics': best_A_120_test.to_dict()},
    'strict_A_30_vs_60_macro_f1_max_absolute_delta': a_30_60_delta,
    'sensitivity_for_selected_A_setting': selected_sensitivity,
    'weakest_classes_in_best_group_A_test': weakest,
    'strongest_classes_in_best_group_A_test': strongest,
    'main_confusion_pairs_in_best_group_A_test': confusion_pairs,
    'near_perfect_warning': near_perfect,
    'risk_notes': [
        'All reported metrics use complete trajectory/sample_id groups; time rows were never randomly split.',
        'Scaler fitting is inside the Logistic Regression pipeline and uses train trajectories only.',
        'If Macro-F1 is near 1.0, treat it as a leakage/confounding signal rather than a generalization claim.',
        'The strict A test Macro-F1 was identical at 30 s and 60 s for every compared model; the early window did not add measurable class separation.',
        '09 identified direct accident/control/radiological variables outside the 38-feature strict set; candidate LVCR is explicitly tested in group B.',
    ],
}
with open(RESULT_ROOT / '10_multi_accident_summary.json', 'w', encoding='utf-8') as handle:
    json.dump(json_safe(summary), handle, ensure_ascii=False, indent=2)

assert (RESULT_ROOT / '10_multi_accident_metrics.csv').exists()
assert (RESULT_ROOT / '10_multi_accident_per_class.csv').exists()
assert (RESULT_ROOT / '10_multi_accident_summary.json').exists()
assert (FIGURE_ROOT / '10_30_vs_60_comparison.png').exists()
assert (FIGURE_ROOT / '10_sensitivity_comparison.png').exists()
assert len(confusion) == len(metrics) * len(CLASS_LABELS) * len(CLASS_LABELS)
assert confusion.groupby(['window_s', 'input_group', 'model', 'eval_split'])['count'].sum().ge(1).all()
print('Best strict group A validation selection:', best_A_validation[['window_s', 'model', 'macro_f1', 'balanced_accuracy']].to_dict())
print('Best strict group A test metrics:', best_A_test[['window_s', 'model', 'accuracy', 'macro_f1', 'balanced_accuracy']].to_dict())
print('Best overall primary test metrics:', best_overall_test[['window_s', 'input_group', 'model', 'accuracy', 'macro_f1', 'balanced_accuracy']].to_dict())
print('FULL MULTI-ACCIDENT CLASSIFICATION PASSED')


Best strict group A validation selection: {'window_s': 30, 'model': 'hist_gradient_boosting', 'macro_f1': 0.19586894586894588, 'balanced_accuracy': 0.27499999999999997}
Best strict group A test metrics: {'window_s': 30, 'model': 'hist_gradient_boosting', 'accuracy': 0.2764227642276423, 'macro_f1': 0.19275498978469274, 'balanced_accuracy': 0.2727272727272727}
Best overall primary test metrics: {'window_s': 30, 'input_group': 'A_strict_38', 'model': 'hist_gradient_boosting', 'accuracy': 0.2764227642276423, 'macro_f1': 0.19275498978469274, 'balanced_accuracy': 0.2727272727272727}
FULL MULTI-ACCIDENT CLASSIFICATION PASSED
